# Week 2 - Custom Evaluation Metric

## Objective

The standard RMSE treats overestimation and underestimation equally. However, in food delivery, underestimating the delivery time is more harmful because customers expect their order to arrive within the predicted time.

The goal of this notebook is to design a custom evaluation metric that penalizes underestimation more than overestimation and compare all four models using both RMSE and the custom metric.

## Assumption

In this project, I assume that underestimating the delivery time is **2 times worse** than overestimating it.

For eg, if the app shows **20 mins** but the delivery actually takes **35 mins**, the customer has to wait longer than expected, which can lead to lower customer satisfaction and make them angry.

On the other hand, if the app shows **35 mins** but the food arrives in **30 mins**, customers are usually satisfied because the order arrives earlier than expected.

Therefore, the custom metric gives a higher penalty to underestimation than overestimation.

## Formula

Let,

- \(y_{true}\) = Actual delivery time
- \(y_{pred}\) = Predicted delivery time

Since we assumed that the penalty of underestimation to be higher, if the predicted time is less than the actual time (underestimation), the squared error is multiplied by **2**.

Otherwise, the normal squared error is used, that is **1**.

\[
Error=
\begin{cases}
2(y_{true}-y_{pred})^2,&\text{if }y_{pred}<y_{true}\\
(y_{true}-y_{pred})^2,&\text{otherwise}
\end{cases}
\]

The final custom metric is

\[
Custom\ RMSE=
\sqrt{\frac{\sum Error}{N}}
\]

In [10]:
# ==========================================================
# Import Required Libraries
# ==========================================================

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure notebook display
pd.set_option("display.max_columns", None)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [11]:
# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("data/food_delivery_analytics_cleaned.csv")

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (15000, 30)


,order_id,city_tier,customer_age,customer_loyalty_score,order_hour,order_day_of_week,order_month,delivery_distance_km,preparation_time_minutes,delivery_time_minutes,estimated_delivery_time,traffic_level_score,weather_severity_score,restaurant_rating,delivery_partner_rating,customer_rating,order_value,delivery_fee,discount_amount,tip_amount,final_amount_paid,number_of_items,cancellation_flag,delayed_delivery_flag,refund_flag,promo_code_used,premium_customer_flag,festival_or_weekend_flag,delivery_partner_experience_years,delivery_efficiency_score
0,5a87c6ab-e4a8-44ef-8852-f7a63e3b3943,2,21,4.957522,20,6,6,14.117144,23,76,68,5.4,2.6,4.3,4.1,2.9,100.000000,14.592602,22.969359,19.940541,111.563784,4,False,False,True,False,False,True,9,71.1
1,8eab78a5-a5c5-41d7-9a5f-5779ee5f2d3d,1,63,38.744721,0,2,2,9.177354,16,34,40,1.0,1.6,4.0,4.2,3.4,100.000000,4.391720,4.434405,16.101949,116.059264,7,False,False,False,True,False,False,12,100.0
2,1338cc5b-e5cf-419f-a7c9-4a2577608715,1,68,45.170997,9,2,11,34.753921,41,152,142,8.3,4.1,4.4,4.2,3.7,100.000000,9.006407,14.979691,17.681454,111.708170,12,False,False,False,False,False,False,10,34.4
3,5277f2fb-b1b3-4f58-9975-12f8f1b71421,2,30,10.573003,6,6,5,26.596184,9,93,93,3.2,8.8,4.8,4.6,4.1,145.113442,11.407034,14.813044,11.593912,153.301345,7,False,False,False,False,True,True,1,45.0
4,df159a97-3a78-4b08-a524-ec433c75b670,2,60,58.284620,12,1,9,26.204152,49,141,134,7.5,9.5,4.0,3.9,4.6,100.000000,11.572232,13.716308,13.272883,111.128807,2,True,False,False,False,False,False,6,25.1


In [12]:
# ==========================================================
# Prepare Clean Dataset
# ==========================================================

features_to_remove = [
    "estimated_delivery_time",
    "delivery_efficiency_score",
    "delayed_delivery_flag",
    "refund_flag",
    "cancellation_flag"
]

X = df.drop(
    columns=[
        "order_id",
        "delivery_time_minutes"
    ] + features_to_remove
)

y = df["delivery_time_minutes"]

print(X.shape)

(15000, 23)


In [13]:
# Convert Boolean columns to integers

bool_columns = X.select_dtypes(include="bool").columns

X[bool_columns] = X[bool_columns].astype(int)

In [14]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [16]:
# ==========================================================
# Custom Evaluation Metric Function
# ==========================================================

import numpy as np

def custom_metric(y_true, y_pred):

    errors = []

    for actual, predicted in zip(y_true, y_pred):

        # Squared Error
        error = (actual - predicted) ** 2

        # Apply extra penalty for underestimation
        if predicted < actual:
            error = error * 2

        errors.append(error)

    return np.sqrt(np.mean(errors))

In [17]:
# ==========================================================
# Testing the Custom Metric
# ==========================================================

y_true = [35, 40, 20]
y_pred = [20, 45, 25]

print("Custom Metric:", round(custom_metric(y_true, y_pred), 2))

Custom Metric: 12.91


In [19]:
# ==========================================================
# Import Models
# ==========================================================

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import root_mean_squared_error
import pandas as pd

In [20]:
# ==========================================================
# Evaluate Models
# ==========================================================

results = []

def evaluate_model(model, model_name):

    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # RMSE
    rmse = root_mean_squared_error(y_test, y_pred)

    # Custom Metric
    custom_score = custom_metric(y_test, y_pred)

    # Store results
    results.append({
        "Model": model_name,
        "RMSE": round(rmse, 2),
        "Custom Metric": round(custom_score, 2)
    })

In [21]:
# ==========================================================
# Evaluate All Models
# ==========================================================

# Linear Regression
evaluate_model(
    LinearRegression(),
    "Linear Regression"
)

# Ridge Regression
evaluate_model(
    Ridge(alpha=100),
    "Ridge Regression"
)

# Random Forest
evaluate_model(
    RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest"
)

# Tuned XGBoost
evaluate_model(
    XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8
    ),
    "Tuned XGBoost"
)

In [22]:
# ==========================================================
# Model Comparison
# ==========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Custom Metric"
).reset_index(drop=True)

results_df

,Model,RMSE,Custom Metric
0,Linear Regression,8.07,9.86
1,Ridge Regression,8.07,9.86
2,Tuned XGBoost,8.17,10.00
3,Random Forest,8.57,10.47


## Observation

All 4 models were evaluated using both RMSE and the custom metric. The custom metric produced higher values than RMSE because it gives extra penalty for underestimating the delivery time.

The ranking of the models did not change. Linear Regression and Ridge Regression remained the best-performing models, followed by Tuned XGBoost and Random Forest. This shows that Linear Regression not only has the lowest prediction error but also makes fewer costly underestimations compared to the other models.

#Justification

RMSE is a useful evaluation metric because it measures the overall prediction error but, it treats overestimation and underestimation equally. In a food delivery business, customers expect their order to arrive at thr earliest or within the promised time. If the delivery is delayed beyond the predicted time, it can lead to complaints, poor rating, and loos of customers trust eventually the customer becomes angry.

My custom metric solves this issue by giving a high (2) penalty to underestimation while keeping the normal penalty (1) for overestimation. This makes the evaluation more aligned with the real business objective of reducing late deliveries. 

The ranking of the models didn't change when compared with the previous week 2 file, the custom metric provides better insight abt how each model performs from a customer and business perspective Therefore, it is more suitable than RMSE for evaluating delivery time prediction models in real-world food delivery applications.

## Custom Metric Summary

The custom evaluation metric was designed to give higher penalty to underestimating delivery time, because late deliveries have a greater negative impact on customer satisfaction than early deliveries.

 All 4 models were evaluated using both RMSE and custom metric. The model ranking remained unchanged, with Linera Regression and Ridge Regression were at the top like before. This shows that the best model based on RMSE also preforms well when business impact is taken into account